In [3]:
import nltk
import pandas as pd
import re
import joblib
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# -----------------------------
# Download NLTK data
# -----------------------------
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

# -----------------------------
# Load Data
# -----------------------------
train_df = pd.read_json("train.json")
test_df = pd.read_json("test.json")

# -----------------------------
# Dedup BEFORE preprocessing
# -----------------------------
train_df = train_df.dropna(subset=["text"]).drop_duplicates(subset=["text"])
test_df = test_df.dropna(subset=["text"]).drop_duplicates(subset=["text"])

# -----------------------------
# Preprocessing
# -----------------------------
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [stemmer.stem(w) for w in tokens]
    return " ".join(tokens)

train_df["text"] = train_df["text"].apply(preprocess)
test_df["text"] = test_df["text"].apply(preprocess)

# -----------------------------
# Features / Labels
# -----------------------------
X_train = train_df["text"]
y_train = train_df["label"]
X_test = test_df["text"]
y_test = test_df["label"]

# -----------------------------
# Pipeline (TF-IDF instead of CountVectorizer)
# -----------------------------
pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(min_df=1, max_df=0.9, ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000))
])

# -----------------------------
# Grid Search (expanded alpha range)
# -----------------------------
param_grid = {
    "classifier__C": [8.0, 10.0, 13.0, 15.0, 17.0, 19.0]  # C is inverse of regularization
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1_weighted"
)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
print("Best params:", grid_search.best_params_)

# -----------------------------
# Save model
# -----------------------------
joblib.dump(best_model, "sentiment_model_lr.joblib")
print("Model saved as sentiment_model.joblib")

# -----------------------------
# Predict on test dataset
# -----------------------------
test_predictions = best_model.predict(X_test)
test_probabilities = best_model.predict_proba(X_test)

# -----------------------------
# Evaluation
# -----------------------------
print("Accuracy:", accuracy_score(y_test, test_predictions))
print("\nClassification Report:\n", classification_report(y_test, test_predictions))

# -----------------------------
# Show some predictions
# -----------------------------
for i in range(5):
    text = test_df["text"].iloc[i]
    pred_label = "Positive" if test_predictions[i] == 1 else "Negative"
    print("Review:", text[:200], "...")
    print("Prediction:", pred_label)
    print("Positive Prob:", test_probabilities[i][1])
    print("Negative Prob:", test_probabilities[i][0])
    print("-" * 60)

# -----------------------------
# Load model (example)
# -----------------------------
# loaded_model = joblib.load("sentiment_model.pkl")
# loaded_model.predict(["some new review text"])

[nltk_data] Downloading package punkt to /home/noob/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/noob/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/noob/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Best params: {'classifier__C': 10.0}
Model saved as sentiment_model.joblib
Accuracy: 0.8886738437966211

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.89      0.89     12361
           1       0.89      0.89      0.89     12440

    accuracy                           0.89     24801
   macro avg       0.89      0.89      0.89     24801
weighted avg       0.89      0.89      0.89     24801

Review: went saw movi last night coax friend mine admit reluct see knew ashton kutcher abl comedi wrong kutcher play charact jake fischer well kevin costner play ben randal profession sign good movi toy emot  ...
Prediction: Positive
Positive Prob: 0.9283397353277034
Negative Prob: 0.07166026467229658
------------------------------------------------------------
Review: actor turn director bill paxton follow promis debut gothic horror frailti famili friendli sport drama u open young american caddi rise humbl background play bristish idol d